In [ ]:
!pip install -q transformers datasets accelerate evaluate librosa soundfile scikit-learn pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import torch
import librosa
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
ds_train = load_dataset("szzs1693/coswara-data", "audio", split="train")
ds_val   = load_dataset("szzs1693/coswara-data", "audio", split="val")
ds_test  = load_dataset("szzs1693/coswara-data", "audio", split="test")

print("Original sizes:", len(ds_train), len(ds_val), len(ds_test))

In [ ]:
COUGH_TYPES = ["cough-heavy", "cough-shallow"]

def filter_cough(ds):
    return ds.filter(lambda x: x["audio_type"] in COUGH_TYPES)

ds_train = filter_cough(ds_train)
ds_val   = filter_cough(ds_val)
ds_test  = filter_cough(ds_test)

print("Filtered sizes:", len(ds_train), len(ds_val), len(ds_test))
print("Train types:", set(ds_train["audio_type"]))

In [ ]:
def map_cough_label(audio_type):
    if audio_type == "cough-heavy":
        return "heavy"
    elif audio_type == "cough-shallow":
        return "shallow"
    else:
        raise ValueError(f"Unexpected audio_type: {audio_type}")

def build_metadata(ds):
    records = []
    skipped_none = 0
    skipped_empty = 0
    skipped_error = 0

    for i, sample in enumerate(ds):
        try:
            audio_obj = sample["audio"]

            if audio_obj is None:
                skipped_none += 1
                continue

            decoded = audio_obj.get_all_samples()

            if hasattr(decoded, "data"):
                y = decoded.data
            else:
                y = decoded

            if hasattr(y, "cpu"):
                y = y.cpu().numpy()

            y = np.asarray(y, dtype=np.float32)

            if y.ndim == 2:
                y = y.mean(axis=0)

            if y is None or len(y) == 0:
                skipped_empty += 1
                continue

            label_name = map_cough_label(sample["audio_type"])

            records.append({
                "index": i,
                "audio_type": sample["audio_type"],
                "label_name": label_name
            })

        except Exception as e:
            skipped_error += 1
            continue

    df = pd.DataFrame(records)

    print("usable:", len(df))
    print("skipped_none:", skipped_none)
    print("skipped_empty:", skipped_empty)
    print("skipped_error:", skipped_error)

    return df

train_df = build_metadata(ds_train)
val_df   = build_metadata(ds_val)
test_df  = build_metadata(ds_test)

In [ ]:
print("Train distribution:")
print(train_df["label_name"].value_counts())

print("\nVal distribution:")
print(val_df["label_name"].value_counts())

print("\nTest distribution:")
print(test_df["label_name"].value_counts())

In [ ]:
CLASS_NAMES = ["heavy", "shallow"]

label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}

train_df["label"] = train_df["label_name"].map(label2id)
val_df["label"]   = val_df["label_name"].map(label2id)
test_df["label"]  = test_df["label_name"].map(label2id)

print(train_df.head())

In [ ]:
train_hf = Dataset.from_pandas(train_df[["index", "label"]], preserve_index=False)
val_hf   = Dataset.from_pandas(val_df[["index", "label"]], preserve_index=False)
test_hf  = Dataset.from_pandas(test_df[["index", "label"]], preserve_index=False)

print(train_hf)
print(val_hf)
print(test_hf)

In [ ]:
CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(CHECKPOINT)

TARGET_SR = feature_extractor.sampling_rate
MAX_SECONDS = 5
MAX_LENGTH = TARGET_SR * MAX_SECONDS

print("Target SR:", TARGET_SR)
print("Max length:", MAX_LENGTH)

In [ ]:
def standardize_audio_from_sample(sample, target_sr=TARGET_SR, max_length=MAX_LENGTH):
    audio_obj = sample["audio"]
    decoded = audio_obj.get_all_samples()

    if hasattr(decoded, "data"):
        y = decoded.data
    else:
        y = decoded

    if hasattr(y, "cpu"):
        y = y.cpu().numpy()

    y = np.asarray(y, dtype=np.float32)

    if y.ndim == 2:
        y = y.mean(axis=0)

    sr = getattr(decoded, "sample_rate", 16000)

    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

    y, _ = librosa.effects.trim(y, top_db=20)

    if len(y) == 0:
        y = np.zeros(max_length, dtype=np.float32)

    if len(y) < max_length:
        y = np.pad(y, (0, max_length - len(y)))
    else:
        y = y[:max_length]

    return y.astype(np.float32)

In [ ]:
def preprocess_train(example):
    sample = ds_train[int(example["index"])]
    y = standardize_audio_from_sample(sample)

    features = feature_extractor(
        y,
        sampling_rate=TARGET_SR,
        max_length=MAX_LENGTH,
        truncation=True
    )

    return {
        "input_values": features["input_values"][0],
        "label": example["label"]
    }

def preprocess_val(example):
    sample = ds_val[int(example["index"])]
    y = standardize_audio_from_sample(sample)

    features = feature_extractor(
        y,
        sampling_rate=TARGET_SR,
        max_length=MAX_LENGTH,
        truncation=True
    )

    return {
        "input_values": features["input_values"][0],
        "label": example["label"]
    }

def preprocess_test(example):
    sample = ds_test[int(example["index"])]
    y = standardize_audio_from_sample(sample)

    features = feature_extractor(
        y,
        sampling_rate=TARGET_SR,
        max_length=MAX_LENGTH,
        truncation=True
    )

    return {
        "input_values": features["input_values"][0],
        "label": example["label"]
    }

In [ ]:
train_hf = train_hf.map(preprocess_train, remove_columns=["index"])
val_hf   = val_hf.map(preprocess_val, remove_columns=["index"])
test_hf  = test_hf.map(preprocess_test, remove_columns=["index"])

train_hf.set_format(type="torch", columns=["input_values", "label"])
val_hf.set_format(type="torch", columns=["input_values", "label"])
test_hf.set_format(type="torch", columns=["input_values", "label"])

print(train_hf[0].keys())

In [ ]:
model = ASTForAudioClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(CLASS_NAMES),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Device:", device)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [ ]:
OUTPUT_DIR = "/content/ast_coswara_cough_heavy_shallow"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-6,
    weight_decay=0.05,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
train_result = trainer.train()

trainer.save_model(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)

print("Training complete.")
print("Saved locally to:", OUTPUT_DIR)

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_hf)
print(test_metrics)

In [ ]:
pred_output = trainer.predict(test_hf)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

In [ ]:
pred_output = trainer.predict(test_hf)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

In [ ]:
import time

model.eval()

sample = test_hf[0]["input_values"].unsqueeze(0).to(device)

for _ in range(10):
    with torch.no_grad():
        _ = model(input_values=sample)

if device == "cuda":
    torch.cuda.synchronize()

times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(input_values=sample)
    if device == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()
    times.append((end - start) * 1000)

print(f"Average latency: {np.mean(times):.2f} ms")
print(f"Std latency: {np.std(times):.2f} ms")
print(f"Min latency: {np.min(times):.2f} ms")
print(f"Max latency: {np.max(times):.2f} ms")

In [ ]:
train_df.to_csv("coswara_cough_heavy_shallow_train.csv", index=False)
val_df.to_csv("coswara_cough_heavy_shallow_val.csv", index=False)
test_df.to_csv("coswara_cough_heavy_shallow_test.csv", index=False)

print("CSV files saved!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR_DRIVE = "/content/drive/MyDrive/ast_coswara_cough_heavy_shallow"

trainer.save_model(OUTPUT_DIR_DRIVE)
feature_extractor.save_pretrained(OUTPUT_DIR_DRIVE)

print("Saved to:", OUTPUT_DIR_DRIVE)

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/ast_coswara_cough_heavy_shallow"))